# 03 — Official Test Evaluation & XGBoost Explainability (FD001–FD004)

This notebook loads the final pooled model saved by Notebook 02, predicts each official test engine's RUL at its last observed cycle, and evaluates performance separately by subset and overall. The official test labels are used here for evaluation only.

Because the fitted training target is capped at 125 cycles, report both **uncapped official RUL metrics** and **metrics with both target and prediction clipped to 125**. Do not quote only the better number without explaining the definition.

In [ ]:
from pathlib import Path
import zipfile
import shutil
import requests
import numpy as np
import pandas as pd

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
REQUIRED_FILES = [f"{kind}_{subset}.txt" for subset in SUBSETS for kind in ("train", "test", "RUL")]
DATASET_URLS = [
    "https://data.nasa.gov/docs/legacy/CMAPSSData.zip",
    "https://zenodo.org/records/15346912/files/CMAPSSData.zip?download=1",
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
]

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def find_dataset_file(filename):
    """Look in data/raw first, then an extracted CMAPSSData folder, then project root."""
    candidates = [RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def ensure_full_dataset(download_if_missing=True):
    missing = [name for name in REQUIRED_FILES if find_dataset_file(name) is None]
    if not missing:
        print("All 12 C-MAPSS files found across data/raw / CMAPSSData / project root.")
        return
    if not download_if_missing:
        raise FileNotFoundError(
            "Missing C-MAPSS files: " + ", ".join(missing) +
            ". Copy all train_FD00x.txt, test_FD00x.txt and RUL_FD00x.txt files into data/raw/ or extract CMAPSSData.zip into the project root."
        )
    archive_path = RAW_DIR / "CMAPSSData.zip"
    last_error = None
    downloaded = False
    for url in DATASET_URLS:
        try:
            print(f"Attempting to download the full C-MAPSS archive from {url.split('/')[2]} ...")
            temporary_path = archive_path.with_suffix(".zip.part")
            with requests.get(url, stream=True, timeout=(20, 180)) as response:
                response.raise_for_status()
                with open(temporary_path, "wb") as out:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            out.write(chunk)
            if not zipfile.is_zipfile(temporary_path):
                temporary_path.unlink(missing_ok=True)
                raise ValueError("The server response was not a valid ZIP archive.")
            temporary_path.replace(archive_path)
            downloaded = True
            break
        except Exception as exc:
            last_error = exc
            print(f"Download attempt failed: {type(exc).__name__}: {exc}")
    if not downloaded:
        raise RuntimeError(
            "Automatic download did not succeed. The full archive is available from the NASA PCoE repository: "
            "https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/ . "
            "Download '6. Turbofan Engine Degradation Simulation Data Set', extract it, and copy all 12 required .txt files to data/raw/. "
            f"Last error: {last_error}"
        )
    with zipfile.ZipFile(archive_path) as archive:
        members = {Path(member).name: member for member in archive.namelist() if not member.endswith("/")}
        not_in_archive = [name for name in REQUIRED_FILES if name not in members]
        if not_in_archive:
            raise FileNotFoundError(f"Archive is missing expected files: {not_in_archive}")
        for filename in REQUIRED_FILES:
            with archive.open(members[filename]) as src, open(RAW_DIR / filename, "wb") as dst:
                shutil.copyfileobj(src, dst)
    print(f"Extracted all C-MAPSS files to {RAW_DIR}")


def load_cmapss_file(filename):
    path = find_dataset_file(filename)
    if path is None:
        raise FileNotFoundError(f"{filename} was not found. Run ensure_full_dataset() or follow README.md.")
    frame = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python")
    frame = frame.dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected column count in {path}: {frame.shape[1]}")
    return frame


def load_all_subsets(include_test=True):
    train_frames, test_frames, rul_frames = [], [], []
    for subset in SUBSETS:
        train = load_cmapss_file(f"train_{subset}.txt")
        train["subset"] = subset
        train["engine_key"] = subset + "_" + train["unit_nr"].astype(int).astype(str).str.zfill(3)
        train_frames.append(train)
        if include_test:
            test = load_cmapss_file(f"test_{subset}.txt")
            test["subset"] = subset
            test["engine_key"] = subset + "_" + test["unit_nr"].astype(int).astype(str).str.zfill(3)
            test_frames.append(test)
            rul_path = find_dataset_file(f"RUL_{subset}.txt")
            if rul_path is None:
                raise FileNotFoundError(f"RUL_{subset}.txt is missing.")
            rul_vector = pd.read_csv(rul_path, header=None).iloc[:, 0].to_numpy(dtype=float)
            test_ids = np.sort(test["unit_nr"].unique().astype(int))
            if len(test_ids) != len(rul_vector):
                raise ValueError(f"{subset}: {len(test_ids)} test engines but {len(rul_vector)} RUL labels.")
            rul_frames.append(pd.DataFrame({
                "subset": subset,
                "unit_nr": test_ids,
                "engine_key": [f"{subset}_{u:03d}" for u in test_ids],
                "true_rul_cycles": rul_vector,
            }))
    train_all = pd.concat(train_frames, ignore_index=True)
    if not include_test:
        return train_all
    return train_all, pd.concat(test_frames, ignore_index=True), pd.concat(rul_frames, ignore_index=True)


import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ensure_full_dataset(download_if_missing=False)
MODEL_PATH = MODELS_DIR / "xgb_rul_model_all_subsets.joblib"
if not MODEL_PATH.exists():
    raise FileNotFoundError("Full-subset artifact is missing. Run Notebook 02, then Notebook 04 to select/refit the robust model.")
artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
SENSOR_COLS = artifact["sensor_cols"]
SETTING_COLS = artifact["setting_cols"]
FEATURE_COLUMNS = artifact["feature_columns"]
ROLLING_WINDOW = int(artifact["rolling_window"])
RUL_CAP = artifact.get("rul_cap")
RUL_EVAL_CAP = 125  # fixed diagnostic convention, independent of training target cap
print("Loaded pooled model:", artifact["dataset"])
print("Model features:", len(FEATURE_COLUMNS))

def add_regime_features_for_artifact(features, transformer):
    """Apply the saved train-only operating-regime transformer at evaluation time."""
    out = features.copy().reset_index(drop=True)
    scaled = transformer["setting_scaler"].transform(out[SETTING_COLS])
    regime = transformer["kmeans"].predict(scaled).astype(int)
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    group_stats = transformer["group_stats"]
    subset_stats = transformer["subset_stats"]
    global_stats = transformer["global_stats"]
    positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means = np.empty(len(out), dtype=float)
        stds = np.empty(len(out), dtype=float)
        for key, pos in positions.items():
            stats = group_stats.get(key, subset_stats.get(key[0], global_stats))
            mean, sd = stats.get(source_col, global_stats[source_col])
            means[pos] = mean
            stds[pos] = max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    return out


## 1. Predict at the last observed cycle for every official test engine

In [ ]:
def engineer_features(frame, window=ROLLING_WINDOW):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda series: series.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda series: series.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    return features

train_all, test_all, test_rul_all = load_all_subsets(include_test=True)
test_features = engineer_features(test_all)
if artifact.get("feature_mode", "baseline") == "regime_aware":
    test_features = add_regime_features_for_artifact(test_features, artifact["regime_transformer"])
missing_cols = [c for c in FEATURE_COLUMNS if c not in test_features.columns]
if missing_cols:
    raise ValueError(f"Test feature pipeline does not match trained artifact. Missing: {missing_cols}")
X_test_all_rows = test_features[FEATURE_COLUMNS]
test_features["predicted_rul_cycles"] = np.maximum(0.0, model.predict(X_test_all_rows))
test_last = (
    test_features.sort_values(["subset", "unit_nr", "time_cycles"])
    .groupby(["subset", "unit_nr"], as_index=False, sort=False).tail(1)
    .sort_values(["subset", "unit_nr"]).reset_index(drop=True)
)
labels = test_rul_all[["subset", "unit_nr", "true_rul_cycles"]].copy()
results = test_last.merge(labels, on=["subset", "unit_nr"], how="left", validate="one_to_one")
if results["true_rul_cycles"].isna().any():
    raise ValueError("Some official RUL labels did not align to test engines.")
results["predicted_rul_cycles"] = np.maximum(0.0, results["predicted_rul_cycles"])
results["error_pred_minus_true_cycles"] = results["predicted_rul_cycles"] - results["true_rul_cycles"]
results["absolute_error_uncapped_cycles"] = results["error_pred_minus_true_cycles"].abs()
results["true_rul_capped_cycles"] = results["true_rul_cycles"].clip(upper=RUL_EVAL_CAP)
results["predicted_rul_capped_cycles"] = results["predicted_rul_cycles"].clip(upper=RUL_EVAL_CAP)
print("Official test engines evaluated:", len(results))
display(results.groupby("subset").agg(
    test_engines=("engine_key", "nunique"),
    median_true_rul=("true_rul_cycles", "median"),
    median_predicted_rul=("predicted_rul_cycles", "median"),
    uncapped_mae=("absolute_error_uncapped_cycles", "mean"),
).round(3))

## 2. Overall and per-subset test metrics

MAE is intuitive average absolute error; RMSE penalises larger misses more; R² compares predictions with a constant-mean baseline. The NASA-style asymmetric score penalises late RUL predictions more heavily than early predictions. This is a benchmark metric, not a calibrated real-world cost function.

In [ ]:
def nasa_asymmetric_score(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    penalties = np.where(error < 0,
                         np.exp(np.clip(-error / 13.0, 0, 50)) - 1,
                         np.exp(np.clip(error / 10.0, 0, 50)) - 1)
    return float(np.sum(penalties))


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(0.0, np.asarray(y_pred, dtype=float))
    return {
        "n_engines": int(len(y_true)),
        "MAE_cycles": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_cycles": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else None,
        "NASA_asymmetric_score_lower_is_better": nasa_asymmetric_score(y_true, y_pred),
        "early_predictions_pct": float(np.mean(y_pred < y_true) * 100),
        "late_predictions_pct": float(np.mean(y_pred > y_true) * 100),
    }

metrics = {
    "dataset": "NASA PCoE C-MAPSS FD001-FD004",
    "algorithm": "Pooled XGBoost Regressor",
    "evaluation_unit": "last observed cycle per official test engine",
    "n_test_engines": int(len(results)),
    "training_target_cap_cycles": RUL_CAP,
    "training_feature_mode": artifact.get("feature_mode", "baseline"),
    "capped_metric_definition": f"true and predicted RUL clipped to {RUL_EVAL_CAP} cycles for diagnostic comparison",
    "uncapped_official_rul_metrics": calculate_metrics(results["true_rul_cycles"], results["predicted_rul_cycles"]),
    "capped_125_rul_metrics": calculate_metrics(results["true_rul_capped_cycles"], results["predicted_rul_capped_cycles"]),
    "by_subset_uncapped_metrics": {},
    "by_subset_capped_metrics": {},
    "official_test_labels_used_for_fit": False,
}
for subset in SUBSETS:
    part = results.loc[results["subset"] == subset]
    metrics["by_subset_uncapped_metrics"][subset] = calculate_metrics(part["true_rul_cycles"], part["predicted_rul_cycles"])
    metrics["by_subset_capped_metrics"][subset] = calculate_metrics(part["true_rul_capped_cycles"], part["predicted_rul_capped_cycles"])

# Engine-level bootstrap CI for uncapped MAE; these are benchmark-engine sampling intervals.
rng = np.random.default_rng(42)
absolute_errors = results["absolute_error_uncapped_cycles"].to_numpy()
bootstrap_indices = rng.integers(0, len(absolute_errors), size=(2500, len(absolute_errors)))
bootstrap_mae = absolute_errors[bootstrap_indices].mean(axis=1)
ci_low, ci_high = np.quantile(bootstrap_mae, [0.025, 0.975])
metrics["uncapped_MAE_bootstrap_95pct_interval_cycles"] = [float(ci_low), float(ci_high)]
(REPORTS_DIR / "all_subsets_test_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

## 3. Engine-level prediction export and visual error analysis

In [ ]:
def risk_band(rul, critical=15, watch=30):
    if rul <= critical:
        return "Critical"
    if rul <= watch:
        return "Watch"
    return "Normal"

results["predicted_risk_band"] = results["predicted_rul_cycles"].map(risk_band)
results.to_csv(REPORTS_DIR / "all_subsets_test_engine_predictions.csv", index=False)
display(results.sort_values("absolute_error_uncapped_cycles", ascending=False).head(20).round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for subset in SUBSETS:
    part = results.loc[results["subset"] == subset]
    axes[0].scatter(part["true_rul_cycles"], part["predicted_rul_cycles"], alpha=0.7, label=subset)
limit = float(max(results["true_rul_cycles"].max(), results["predicted_rul_cycles"].max(), 1))
axes[0].plot([0, limit], [0, limit], "--", color="gray")
axes[0].set(title="Official test: predicted vs true RUL", xlabel="True RUL (cycles)", ylabel="Predicted RUL (cycles)")
axes[0].legend(title="Subset")
axes[1].hist(results["error_pred_minus_true_cycles"], bins=30, edgecolor="white")
axes[1].axvline(0, linestyle="--", color="gray")
axes[1].set(title="Uncapped test prediction errors", xlabel="Predicted − true RUL (cycles)", ylabel="Test engines")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "all_subsets_test_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
per_subset_mae = pd.DataFrame([
    {"subset": subset, "MAE uncapped": metrics["by_subset_uncapped_metrics"][subset]["MAE_cycles"],
     "MAE capped-125": metrics["by_subset_capped_metrics"][subset]["MAE_cycles"]}
    for subset in SUBSETS
]).melt(id_vars="subset", var_name="metric", value_name="MAE (cycles)")
sns.barplot(data=per_subset_mae, x="subset", y="MAE (cycles)", hue="metric", ax=ax)
ax.set_title("Test MAE by subset: uncapped and capped target definitions")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "all_subsets_mae_by_subset.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Model explainability using XGBoost-native Tree SHAP

Use the built-in XGBoost contribution output instead of relying on the separate `shap` package. `pred_contribs=True` returns one contribution per feature plus a baseline contribution. Contributions should sum to the model's raw prediction up to small floating-point error. They explain model behaviour, not physical causation.

In [ ]:
# Explain the test set's last-cycle rows; 707 rows is small enough for this calculation.
X_explain = results[FEATURE_COLUMNS].copy()
booster = model.get_booster()
dmat = xgb.DMatrix(X_explain, feature_names=FEATURE_COLUMNS)
contributions = booster.predict(dmat, pred_contribs=True)
shap_values = contributions[:, :-1]
base_values = contributions[:, -1]
raw_predictions = model.predict(X_explain)
max_additivity_error = float(np.max(np.abs(contributions.sum(axis=1) - raw_predictions)))
print(f"Explained engines: {len(X_explain)}")
print(f"Features explained: {shap_values.shape[1]}")
print(f"Maximum contribution additivity error: {max_additivity_error:.8f} cycles")

global_importance = pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURE_COLUMNS, name="mean_absolute_shap").sort_values(ascending=False)
global_importance.to_csv(REPORTS_DIR / "all_subsets_shap_global_importance.csv")
display(global_importance.head(20).to_frame())

top_global = global_importance.head(15).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_global.index, top_global.values)
ax.set(title="Global XGBoost Tree SHAP importance (all official test engines)", xlabel="Mean absolute SHAP contribution (RUL cycles)", ylabel="Feature")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "all_subsets_shap_global_importance.png", dpi=150, bbox_inches="tight")
plt.show()

worst_idx = int(np.argmax(results["absolute_error_uncapped_cycles"].to_numpy()))
local = pd.Series(shap_values[worst_idx], index=FEATURE_COLUMNS)
top_local = local.loc[local.abs().nlargest(15).index].sort_values()
worst_row = results.iloc[worst_idx]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_local.index, top_local.values)
ax.axvline(0, linestyle="--", linewidth=1, color="gray")
ax.set(title=f"Local SHAP contributions — {worst_row['subset']} engine {int(worst_row['unit_nr'])}", xlabel="Contribution to predicted RUL (cycles)", ylabel="Feature")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "all_subsets_shap_worst_engine.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Worst absolute-error engine: {worst_row['subset']} / unit {int(worst_row['unit_nr'])}")
print(f"Predicted RUL: {worst_row['predicted_rul_cycles']:.2f} | True RUL: {worst_row['true_rul_cycles']:.2f} cycles")
print("Global and local SHAP contribution plots saved.")

## Interview-ready interpretation checklist

1. Why this is regression and why RUL is measured in cycles.
2. Why the model is pooled across four simulated regimes and uses subset one-hot features.
3. How `engine_key` prevents engine IDs overlapping between subsets from causing leakage.
4. Why rolling features use only past/current observations.
5. Why validation and CV split whole engines, not rows.
6. How the 125-cycle cap helps focus learning and limits predictions above 125.
7. Why both uncapped and capped test metrics are reported.
8. Why MAE/RMSE, R² and the asymmetric score communicate different kinds of error.
9. How native XGBoost Tree SHAP contribution values reconstruct predictions and why they are not causal.
10. Why simulated benchmark performance must not be claimed as validated real aircraft maintenance performance.